In [1]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord', 'hackle_events']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")
            
            
# 데이트타임형으로 변환 및 기간 전처리
def set_datetime(df, column):
    df[column] =  pd.to_datetime(df[column])
    print(f'✅ {column}데이트 타입 형변환 및 기간 전처리 완료')
    
    

            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


In [2]:
hackle_events = get_df('hackle', 'hackle_events')

In [3]:
hackle_properties = get_df('hackle', 'hackle_properties')

In [18]:
try:
    hackle_properties['user_id'] = hackle_properties['user_id'].astype('str')
    issue = hackle_properties[~hackle_properties['user_id'].apply(lambda x: x.isdigit())]
    hackle_properties = hackle_properties[hackle_properties['user_id'].apply(lambda x: x.isdigit())]
    hackle_properties = hackle_properties[['session_id', 'user_id']]
except Exception as e:
    print(f'{e}')

In [23]:
(hackle_events['event_id'] == hackle_events['id']).all()
hackle_events = hackle_events.drop(columns='id')

In [25]:
hackle_events = hackle_events[['event_id', 'session_id', 'event_key', 'event_datetime', 'item_name', 'page_name', 'friend_count', 'votes_count', 'heart_balance' ,'question_id']]

In [29]:
merge_df = pd.merge(hackle_events, hackle_properties, on='session_id', how='left')

In [31]:
merge_df = merge_df[['event_id', 'user_id', 'event_key' ,'event_datetime', 'item_name' ,'page_name', 'friend_count' ,'votes_count' ,'heart_balance','question_id', 'session_id']]

In [33]:
merge_df = merge_df.sort_values(by=['user_id', 'event_datetime'])

,event_id,user_id,event_key,event_datetime,item_name,page_name,friend_count,votes_count,heart_balance,question_id,session_id
4647816,39abbfb0-c4a9-4541-8eb6-dcbc15999fca,1000000,$session_start,2023-07-31 02:35:55,,,41.0,62.0,735.0,NaN,d5R93rjsiuOotxzbFc44AAmlvz93
7255596,59f8a999-d137-4b22-8326-1f9161cfe7a7,1000000,launch_app,2023-07-31 02:35:55,,,41.0,62.0,735.0,NaN,d5R93rjsiuOotxzbFc44AAmlvz93
5002469,3e0fd384-5d60-4345-96ec-cfec8512a713,1000009,$session_start,2023-07-19 21:01:30,,,45.0,281.0,4780.0,NaN,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2
5002470,3e0fd384-5d60-4345-96ec-cfec8512a713,1000009,$session_start,2023-07-19 21:01:30,,,45.0,281.0,4780.0,NaN,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2
5677951,466b5a3b-f097-4cbe-98a3-93a7503519e0,1000009,launch_app,2023-07-19 21:01:30,,,45.0,281.0,4780.0,NaN,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2
...,...,...,...,...,...,...,...,...,...,...,...
590281,074f6efe-4730-4a53-94f3-4cb2d11538bf,NaN,$session_start,2023-08-10 23:54:37,,,NaN,NaN,NaN,NaN,qs0Zf3gb8oWiABvqEsmDIAPiBQw2
678746,086914fe-52b7-46d3-be35-613dd0927b4e,NaN,launch_app,2023-08-10 23:54:37,,,NaN,NaN,NaN,NaN,qs0Zf3gb8oWiABvqEsmDIAPiBQw2
5869253,48cb0448-9177-4414-88d3-29fe903cbbe2,NaN,launch_app,2023-08-10 23:54:42,,,NaN,NaN,NaN,NaN,iiyacTPZzMWAtXqsY2N0UW0MKVx1
19015703,ec003f11-cb67-4355-9ee3-af803249c999,NaN,$session_start,2023-08-10 23:54:42,,,NaN,NaN,NaN,NaN,iiyacTPZzMWAtXqsY2N0UW0MKVx1
